<a href="https://colab.research.google.com/github/Ganesh2516/TriVitaX_MedTech/blob/main/MEDTECH_USING_XGboost_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd # Import the pandas library
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

display(df.head())

Saving Dengue-Dataset_Processed.csv to Dengue-Dataset_Processed.csv


,Age,Hemoglobin(g/dl),Neutrophils(%),Lymphocytes(%),Monocytes(%),Eosinophils(%),RBC,HCT(%),MCV(fl),MCH(pg),...,Total Platelet Count(/cumm),MPV(fl),PDW(%),PCT(%),Total WBC count(/cumm),Gender_Female,Gender_Male,NLR,Platelet_WBC_Ratio,Result_Encoded
0,21,14.8,48,47,3,2,5,48.00,96.0,29.60,...,112000,10.70,15.40,0.120,5100,0,1,1.021277,21.960784,1
1,30,15.0,47,49,6,3,5,49.80,96.1,28.40,...,96000,10.60,15.80,0.121,4500,0,1,0.959184,21.333333,1
2,51,16.3,41,48,4,5,5,50.10,93.5,31.30,...,184000,10.40,16.40,0.130,6000,0,1,0.854167,30.666667,0
3,26,12.3,46,49,7,5,5,44.00,90.0,30.50,...,167000,8.10,17.10,0.110,5000,1,0,0.938776,33.400000,0
4,35,16.1,45,46,4,4,5,50.53,91.0,29.12,...,155000,10.52,12.34,0.150,4600,0,1,0.978261,33.695652,0


#1.Hemoglobin Prediction(Red Port)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
# --- Change is here ---
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("Dengue-Dataset_Processed.csv")

HEMOGLOBIN_THRESHOLD = 12.0
df['Low_Hemoglobin'] = (df['Hemoglobin(g/dl)'] < HEMOGLOBIN_THRESHOLD).astype(int)

# Define target and features
Y_hemoglobin = df['Low_Hemoglobin']
X_hemoglobin = df.drop(columns=['Hemoglobin(g/dl)', 'Low_Hemoglobin', 'Result_Encoded'])

# Split data
X_train, X_test, Y_train, Y_test = train_test_split(
    X_hemoglobin, Y_hemoglobin, test_size=0.3, random_state=42, stratify=Y_hemoglobin)

# --- Change is here: Use XGBoost Classifier ---
model_hemoglobin = XGBClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=6, # Common depth for XGBoost
    use_label_encoder=False,
    eval_metric='logloss'
)
model_hemoglobin.fit(X_train, Y_train)

Y_pred = model_hemoglobin.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)

print(f"--- Low Hemoglobin Prediction Model (Red Port) ---")
print(f"Prediction Target: Low Hemoglobin (< {HEMOGLOBIN_THRESHOLD} g/dl)")
print(f"Model: XGBoost Classifier")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

--- Low Hemoglobin Prediction Model (Red Port) ---
Prediction Target: Low Hemoglobin (< 12.0 g/dl)
Model: XGBoost Classifier
Accuracy: 93.87%

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       424
           1       0.78      0.21      0.33        33

    accuracy                           0.94       457
   macro avg       0.86      0.60      0.65       457
weighted avg       0.93      0.94      0.92       457



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:33:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


#2. Platelets Prediction (Yellow Port)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("Dengue-Dataset_Processed.csv")

PLATELET_THRESHOLD = 150000
df['Low_Platelets'] = (df['Total Platelet Count(/cumm)'] < PLATELET_THRESHOLD).astype(int)

Y_platelets = df['Low_Platelets']
X_platelets = df.drop(columns=['Total Platelet Count(/cumm)', 'Low_Platelets', 'Result_Encoded'])

X_train, X_test, Y_train, Y_test = train_test_split(
    X_platelets, Y_platelets, test_size=0.3, random_state=42, stratify=Y_platelets)

model_platelets = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, min_samples_leaf=5)
model_platelets.fit(X_train, Y_train)

Y_pred = model_platelets.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)

print(f"--- Low Platelets Prediction Model (Yellow Port) ---")
print(f"Prediction Target: Thrombocytopenia (< {PLATELET_THRESHOLD} /cumm)")
print(f"Model: Random Forest Classifier")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

--- Low Platelets Prediction Model (Yellow Port) ---
Prediction Target: Thrombocytopenia (< 150000 /cumm)
Model: Random Forest Classifier
Accuracy: 94.09%

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       263
           1       0.97      0.89      0.93       194

    accuracy                           0.94       457
   macro avg       0.95      0.93      0.94       457
weighted avg       0.94      0.94      0.94       457

